# Get the title

In [ ]:
import requests
import json
import time
from datetime import datetime
from tqdm import tqdm

def crawl_new_wikipedia_pages(start_date="2026-01-01T00:00:00Z", 
                               end_date="2026-03-07T00:00:00Z",
                               min_size=1500):
    """
    Crawl newly created Wikipedia pages using the recentchanges API.
    This gets ALL new pages created in the date range, not just those in specific categories.
    Much more diverse than category-based crawling (which is sports-heavy).
    
    Args:
        start_date: Start of date range (ISO format)
        end_date: End of date range (ISO format)
        min_size: Minimum page size in bytes to filter out stubs
    
    Returns:
        List of page info dicts with title, timestamp, size, etc.
    """
    url = "https://en.wikipedia.org/w/api.php"
    headers = {"User-Agent": "Knowledge injection research"}
    
    all_new_pages = []
    params = {
        "action": "query",
        "list": "recentchanges",
        "rctype": "new",  # Only new page creations
        "rcnamespace": 0,  # Main article namespace (excludes Talk, User, etc.)
        "rcstart": end_date,  # Note: rcstart is the END when rcdir=older
        "rcend": start_date,
        "rcdir": "older",
        "rclimit": "max",  # Get maximum per request (500 for users, 5000 for bots)
        "rcprop": "title|timestamp|sizes|user",
        "format": "json"
    }
    
    print(f"Fetching new pages from {start_date} to {end_date}...")
    request_count = 0
    while True:
        response = requests.get(url, params=params, headers=headers)
        data = response.json()
        request_count += 1
        
        if "error" in data:
            print(f"API Error: {data['error']}")
            break
        
        changes = data.get("query", {}).get("recentchanges", [])
        # Filter by size to get substantive articles (not stubs)
        substantive = [p for p in changes if p.get("newlen", 0) >= min_size]
        all_new_pages.extend(substantive)
        
        print(f"  Request {request_count}: fetched {len(changes)} pages, {len(substantive)} substantive, total: {len(all_new_pages)}")
        
        if "continue" in data:
            params["rccontinue"] = data["continue"]["rccontinue"]
            time.sleep(0.1)  # Be nice to the API
        else:
            break
    
    print(f"\nTotal new substantive pages (>= {min_size} bytes): {len(all_new_pages)}")
    return all_new_pages

# Crawl new pages created in 2026
# Adjust end_date to current date as needed
new_pages = crawl_new_wikipedia_pages(
    start_date="2026-01-01T00:00:00Z",
    end_date="2026-03-07T00:00:00Z",  # Update this to current date
    min_size=1500  # Filter out stubs, keep pages with at least 1.5KB of content
)

# Extract page titles (for compatibility with rest of notebook)
all_pages = set(p["title"] for p in new_pages)
print(f"\nTotal unique page titles: {len(all_pages)}")

# Show some examples
print("\nSample of crawled pages:")
for p in list(new_pages)[:10]:
    print(f"  - {p['title']} ({p['newlen']} bytes, created {p['timestamp']})")


In [ ]:
len(all_pages)

In [ ]:
# Save all_pages to a JSON file
import json

# Convert set to list for JSON serialization
pages_list = list(all_pages)

# Save to JSON file
with open('all_pages_2026.json', 'w', encoding='utf-8') as f:
    json.dump(pages_list, f, indent=2, ensure_ascii=False)

print(f"Saved {len(pages_list)} pages to all_pages_2026.json")


# Fetch articles

In [ ]:
import requests
import json
from datetime import datetime
import mwparserfromhell
from tqdm import tqdm
import pickle
import time

def remove_template(wikicode, node):
    if isinstance(node, mwparserfromhell.nodes.template.Template) and (node.name == 'Nihongo foot' or node.name == 'nihongo'):
        wikicode.replace(node, mwparserfromhell.nodes.text.Text(node.params[0]))
        
    elif isinstance(node, mwparserfromhell.nodes.tag.Tag) \
        and len(node.contents.nodes) > 0 \
        and isinstance(node.contents.nodes[0], mwparserfromhell.nodes.template.Template) \
        and node.contents.nodes[0].name == 'lang':
        wikicode.replace(node, mwparserfromhell.nodes.text.Text(node.contents.nodes[0].params[-1]))
        
    elif isinstance(node, mwparserfromhell.nodes.tag.Tag) \
        and len(node.contents.nodes) == 1 \
        and isinstance(node.contents.nodes[0], mwparserfromhell.nodes.text.Text):
        wikicode.replace(node, node.contents.nodes[0])
        
    elif isinstance(node, mwparserfromhell.nodes.tag.Tag) \
        and isinstance(node.contents, mwparserfromhell.wikicode.Wikicode):
        wikicode.replace(node, mwparserfromhell.nodes.text.Text(node.contents.strip_code()))
        
    elif not (isinstance(node, mwparserfromhell.nodes.text.Text) 
            or isinstance(node, mwparserfromhell.nodes.tag.Tag)
            or isinstance(node, mwparserfromhell.nodes.wikilink.Wikilink)):
        wikicode.remove(node)
        
    elif isinstance(node, mwparserfromhell.nodes.tag.Tag):
        wikicode.remove(node)
        
    return wikicode

url = "https://en.wikipedia.org/w/api.php"

# Load pages - use the list from the previous cell if available, otherwise load from file
try:
    pages_list = list(all_pages)
    print(f"Using {len(pages_list)} pages from previous cell")
except NameError:
    with open('all_pages_2026.json', 'r') as f:
        pages_list = json.load(f)
    print(f"Loaded {len(pages_list)} pages from all_pages_2026.json")

events_after_2026 = {}

# Since we used recentchanges API with date filtering, all pages are already
# guaranteed to be created after 2026-01-01. No need to check creation dates again!
# This makes the content fetching much faster (1 API call per page instead of 2).

for wikipedia_article in tqdm(pages_list):
    params = {
        "action": "query",
        "format": "json",
        "titles": f"{wikipedia_article}",
        "prop": "revisions",
        "rvlimit": "1",
        "rvprop": "content"
    }
    headers = {
        "User-Agent": "Get events in 2026",
        "Authorization": "Bearer " + os.getenv("WIKIPEDIA_API_TOKEN", "YOUR_TOKEN_HERE")
    }
    
    try:
        response = requests.get(url, params=params, headers=headers)
        data = response.json()
    except Exception as e:
        print(f"Error fetching {wikipedia_article}: {e}")
        continue
    
    try:
        page_data = list(data['query']['pages'].values())[0]
        if 'revisions' not in page_data:
            continue
        text = page_data['revisions'][0]['*']
    except (KeyError, IndexError) as e:
        print(f"Error parsing {wikipedia_article}: {e}")
        continue
    
    wikicode = mwparserfromhell.parse(text)
    for node in wikicode.get_sections()[0].nodes:
        wikicode = remove_template(wikicode, node)
        
    parsed_text = wikicode.get_sections()[0].strip_code()  
    parsed_text = '\n'.join([line for line in parsed_text.split('\n') if not (line.startswith('thumb|'))])
    
    if "Category:" not in parsed_text:
        events_after_2026[wikipedia_article] = parsed_text

print(f"\nCollected {len(events_after_2026)} articles with content")

# Save the results
with open('events_after_2026.json', 'w') as f:
    json.dump(events_after_2026, f, indent=2)


In [ ]:
events_after_2026

# Filter by token numbers

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct")

filtered_wiki_dataset = []

for (title, text) in zip(events_after_2026.keys(), events_after_2026.values()):
    token_length = len(tokenizer.encode(text))-1
    if 65 <= token_length <= 95:
        # Check for title overlap with existing items
        current_title_words = set(title.lower().split())
        has_overlap = False
        
        for existing_item in filtered_wiki_dataset:
            existing_title_words = set(existing_item["title"].lower().split())
            overlap_count = len(current_title_words.intersection(existing_title_words))
            if overlap_count >= 3 or "PFL" in title or "UFC" in title or " list" in text:
                has_overlap = True
                break
        
        if has_overlap:
            continue
        filtered_wiki_dataset.append({"title": title, "text": text})

with open('filtered_wiki_dataset_2026.json', 'w') as f:
    json.dump(filtered_wiki_dataset, f, indent=4)
    
len(filtered_wiki_dataset)

In [ ]:
filtered_wiki_dataset

# Clean articles

In [ ]:
import pickle
import json
import os
import json
import requests
import time
from tqdm import tqdm

with open('filtered_wiki_dataset_2026.json', 'r') as f:
    events_after_2026 = json.load(f)
    

api_key = os.getenv("HARVARD_API_KEY", "")


def api_call(prompt):
    url = "https://go.apis.huit.harvard.edu/ais-openai-direct-limited-schools/v1/chat/completions"
    payload = json.dumps({
        "model": "gpt-5-mini",
        # "model": "o3-mini-2025-01-31",
        # "model": "gpt-4.1-mini-2025-04-14",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 1
    })
    headers = {
        'Content-Type': 'application/json',
        'api-key': api_key
    }
    
    max_tries = 20
    tries = 0
    while tries < max_tries:
        try:
            response = requests.request("POST", url, headers=headers, data=payload)
            response_dict = json.loads(response.text)
            # print(response_dict)
            return response_dict["choices"][0]["message"]["content"]
        except Exception as e:
            print("API Call error, sleeping...", e)
            time.sleep(5)
            tries += 1
    return None

def api_clean_text(text):
    prompt = f"If the following text has format or template issues (e.g. excessive use of punctuations, symbols, template tags), please clean it up. Only return the cleaned text, no other text or comments. If the text is already clean, just return it as is: \n\n {text}"
    return api_call(prompt)


wiki_dataset = []

for event in tqdm(events_after_2026):
    wiki_dataset.append({'title': event['title'], 
                         'text': event['text'],
                         'cleaned_text': api_clean_text(event['text'])})

    with open('filtered_wiki_dataset_cleaned_2026.json', 'w') as f:
        json.dump(wiki_dataset, f, indent=2)


In [ ]:
len(wiki_dataset)

# Save to word

In [ ]:
import pickle
import json
import os
import json
import requests
import time
from tqdm import tqdm

with open('filtered_wiki_dataset_cleaned_2026.json', 'r') as f:
    events_after_2026 = json.load(f)

import random
from docx import Document
# Filter out entries whose cleaned_text contains ":"
events_after_2026 = [entry for entry in events_after_2026 if ":" not in entry.get("cleaned_text", "")]

# 1. Randomly select 1000 entries
random.seed(42)  # For reproducibility
if len(events_after_2026) < 1000:
    selected_entries = events_after_2026.copy()
else:
    selected_entries = random.sample(events_after_2026, 1000)

# 2. Save the 1000 selected entries to JSON
with open('wiki_dataset_2026_1000.json', 'w') as jf:
    json.dump(selected_entries, jf, indent=2)

# 3. Save only title and cleaned_text in Word document
doc = Document()
doc.add_heading('Wiki Dataset Sample (1000 entries)', 0)

for entry in selected_entries:
    title = entry.get("title", "")
    cleaned_text = entry.get("cleaned_text", "")
    doc.add_heading(title, level=1)
    doc.add_paragraph(cleaned_text)

doc.save('wiki_dataset_sample_1000.docx')

# Paraphrase

In [ ]:
# Load the JSON file
import pandas as pd
import requests
import json
import time
import sys
import os
from tqdm import tqdm
import concurrent.futures
import numpy as np

with open('filtered_wiki_dataset_cleaned.json', 'r') as f:
    wiki_dataset = json.load(f)
    print(f"Loaded {len(wiki_dataset)} items from filtered_wiki_dataset_cleaned.json")
    


url = "https://go.apis.huit.harvard.edu/ais-openai-direct-limited-schools/v1/chat/completions"
# url = "https://go.apis.huit.harvard.edu/ais-openai-direct/v1/chat/completions"

api_key = os.getenv("HARVARD_API_KEY", "")

def api_call_o3_paraphrase(prompt):
    # url = "https://go.apis.huit.harvard.edu/ais-openai-direct-limited-schools/v1/chat/completions"
    payload = json.dumps({
        "model": "o3-mini-2025-01-31",
        # "model": "gpt-4.1-mini-2025-04-14",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 1
    })
    headers = {
        'Content-Type': 'application/json',
        'api-key': api_key
    }
    
    max_tries = 20
    tries = 0
    while tries < max_tries:
        try:
            response = requests.request("POST", url, headers=headers, data=payload)
            response_dict = json.loads(response.text)
            # print(response_dict)
            result = response_dict["choices"][0]["message"]["content"]
            paraphrases = json.loads(result)['paraphrases']
            if len(paraphrases) >=10:
                return paraphrases[:10]
            else:
                print("not 10 paraphrases, API Call error, sleeping...")
                print(paraphrases)
                time.sleep(2)
                tries += 1
        except Exception as e:
            print("API Call error, sleeping...", e)
            time.sleep(5)
            tries += 1
    return None


def api_call(prompt):
    url = "https://go.apis.huit.harvard.edu/ais-openai-direct-limited-schools/v1/chat/completions"
    payload = json.dumps({
        "model": "o3-mini-2025-01-31",
        # "model": "gpt-4.1-2025-04-14",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 1
    })
    headers = {
        'Content-Type': 'application/json',
        'api-key': api_key
    }

    max_tries = 20
    tries = 0
    while tries < max_tries:
        try:
            response = requests.request("POST", url, headers=headers, data=payload)
            response_dict = json.loads(response.text)
            # return response_dict["choices"][0]["message"]["content"]
            content = response_dict['choices'][0]['message']['content']
            content = content.strip('`json\n`')
            qa_data = json.loads(content)
            if type(qa_data['qa_data'][0]) == dict:
                qa_data['qa_data'] = [[item['question'], item['answer']] for item in qa_data['qa_data']]
            return qa_data
        except Exception as e:
            print("API Call error, sleeping...", e)
            time.sleep(5)
            tries += 1
    return None

def GPT_paraphrase_same_order(passage):
    
    prompt = (f"Your task is to paraphrase a text paragraph. "
              f"The paragraph is given below. "
              f"Make sure to keep the same meaning but change the wording. "
              f"Do not change any factual information. "
              f"Strictly do NOT change the word order in which the information is presented. "
              f"Only replace the words or phrases with synonyms, so that ordering of the information is the same. "
              f"Try to keep roughly the same length of the original text. "
              f"Give 9 different paraphrases for each text. "
              f"Return a JSON formatted string with one key, called 'paraphrases', and a list of the ORIGINAL text paragraph along with the 9 paraphrases (so the list has total length 10). The first item should be the original text paragraph. Make sure the list has TEN items. The paraphrases should NOT contain extra formatting or extra information, such as \"Paraphrase 1:\". \n\n"
              f"{passage}")
 
    return api_call_o3_paraphrase(prompt)


def GPT_paraphrase_change_order(passage):
    
    prompt = (f"Your task is to paraphrase a text paragraph. "
              f"The paragraph is given below. "
              f"Make sure to keep the same meaning but change the wording. "
              f"Do not change any factual information. "
              f"Change the word order in which the information is presented. "
              
              f"Think about the order in three levels: word, sentence, and paragraph. \n\n"
              f"An example of changing the word order is: "
              f"Original: The cat and the dog were playing. "
              f"Paraphrase: The dog and the cat were playing. \n\n"
              
              f"An example of changing the sentence order is: "
              f"Original: The cat was chasing the dog. "
              f"Paraphrase: The dog was being chased by the cat. \n\n"
              
              f"An example of changing the paragraph order is: "
              f"Original: The cat was chasing the dog. Then, the cat got tired. "
              f"Paraphrase: The cat got tired. Before that, the cat was chasing the dog. \n\n"
              
              f"Try to keep roughly the same length of the original text. "
              f"Give 9 different paraphrases for each text. "
              f"Return a JSON formatted string with one key, called 'paraphrases', and a list of the ORIGINAL text paragraph along with the 9 paraphrases (so the list has total length 10). The first item should be the original text paragraph. Make sure the list has TEN items. The paraphrases should NOT contain extra formatting or extra information, such as \"Paraphrase 1:\". \n\n"
              f"{passage}")
 
    return api_call_o3_paraphrase(prompt)

def GPT_QA(passage):
    
    prompt = f"""Your task is to generate 3 question and answer pairs based on a given passage below. Make sure to provide AMPLE context in the question, including information from the original passage as context. Do not mention "according to the passage". The person who is asking the question is supposed to remember the passage, rather than looking at the passage. Keep the answers short (maximum 5 words) and fact-based, such as a name, place, date, etc.. Return a JSON formatted string with one key, called 'qa_data', and a list of (question, answer) tuples.\\n{passage}"""
    qa = api_call(prompt)
    return qa['qa_data']


import concurrent.futures
from threading import Lock
import time

def process_item(item):
    """Process a single item to generate paraphrases"""
    processed_item = item.copy()
    paraphrases_same_order = GPT_paraphrase_same_order(item['cleaned_text'])
    processed_item['paraphrases_same_order'] = paraphrases_same_order
    paraphrases_change_order = GPT_paraphrase_change_order(item['cleaned_text'])
    processed_item['paraphrases_change_order'] = paraphrases_change_order
    qa = GPT_QA(item['cleaned_text'])
    processed_item['qa'] = qa
    return processed_item

# Initialize variables for parallel processing
new_wiki_dataset = []
dataset_lock = Lock()
save_frequency = 10  # Save every 10 processed items

def save_dataset():
    """Save the current dataset to file"""
    with open('filtered_wiki_dataset_cleaned_with_paraphrases.json', 'w') as f:
        json.dump(new_wiki_dataset, f, indent=4)

# Use ThreadPoolExecutor for parallel processing
max_workers = 8  # Adjust based on API rate limits
with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
    # Submit all tasks
    future_to_item = {executor.submit(process_item, item): item for item in wiki_dataset}
    
    processed_count = 0
    for future in concurrent.futures.as_completed(future_to_item):
        try:
            processed_item = future.result()
            
            # Thread-safe append to dataset
            with dataset_lock:
                new_wiki_dataset.append(processed_item)
                processed_count += 1
                print(f"Processed {processed_count}/{len(wiki_dataset)} items")
                
                # Save periodically
                if processed_count % save_frequency == 0:
                    save_dataset()
                    
                    
        except Exception as exc:
            original_item = future_to_item[future]
            print(f"Item generated an exception: {exc}")

# Final save
save_dataset()
print(f"Completed processing {len(new_wiki_dataset)} items")